# Word2Vec based recommender

In [1]:
from utils.ml import data_dir, tokenize, vectorize, load_data, train_test_split
df = load_data(data_dir)
train, test = train_test_split(df)
sequences_train, tokens = tokenize(train)
sequences_test, _ = tokenize(test, test = True, tokens_df=tokens)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 20:55:01 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/28 20:55:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 20:55:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TOKENIZED DF:


+----------+----+-----+---+-----+-------+--------------------+
|session ID|year|month|day|order|country|            sequence|
+----------+----+-----+---+-----+-------+--------------------+
|         1|2008|    4|  1|    1|     29|[194, 147, 49, 89...|
|         3|2008|    4|  1|    1|     21|[89, 52, 149, 140...|
|         5|2008|    4|  1|    1|      9|               [126]|
|         6|2008|    4|  1|    1|      9|[149, 22, 106, 89...|
|        12|2008|    4|  1|    1|     29|[49, 53, 53, 191,...|
|        13|2008|    4|  1|    1|     38|      [164, 209, 36]|
|        15|2008|    4|  1|    1|     21|[164, 209, 36, 27...|
|        16|2008|    4|  1|    1|     29|           [154, 42]|
|        19|2008|    4|  1|    1|     29|[130, 125, 199, 1...|
|        20|2008|    4|  1|    1|     29|      [188, 107, 84]|
+----------+----+-----+---+-----+-------+--------------------+
only showing top 10 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+----------------

+----------+----+-----+---+-----+-------+--------------------+----+
|session ID|year|month|day|order|country|            sequence|next|
+----------+----+-----+---+-----+-------+--------------------+----+
|         4|2008|    4|  1|    1|     21|               [151]| 183|
|         4|2008|    4|  1|    2|     21|          [151, 183]|  52|
|         4|2008|    4|  1|    3|     21|      [151, 183, 52]| 100|
|         8|2008|    4|  1|    1|      9|                [97]|  19|
|         8|2008|    4|  1|    2|      9|            [97, 19]| 199|
|         8|2008|    4|  1|    3|      9|       [97, 19, 199]|  10|
|         8|2008|    4|  1|    4|      9|   [97, 19, 199, 10]|   4|
|         8|2008|    4|  1|    5|      9|[97, 19, 199, 10, 4]| 197|
|         8|2008|    4|  1|    6|      9|[97, 19, 199, 10,...| 136|
|         8|2008|    4|  1|    7|      9|[97, 19, 199, 10,...|  54|
+----------+----+-----+---+-----+-------+--------------------+----+
only showing top 10 rows
TOKEN INFO: 
+---------

### Training word2vec on train set

In [2]:
vocab = vectorize(sequences_train, model_only=True)
test_vectorized = vectorize(sequences_test, return_vocabulary=False)

Word2Vec model not found. Refitting...


+---+----------+----+-----+---+-----+-------+----+--------------------+
| id|session ID|year|month|day|order|country|next|              vector|
+---+----------+----+-----+---+-----+-------+----+--------------------+
|151|         4|2008|    4|  1|    1|     21| 183|[0.10115931183099...|
|151|         4|2008|    4|  1|    2|     21|  52|[0.10115931183099...|
|183|         4|2008|    4|  1|    2|     21|  52|[-0.0253764074295...|
|151|         4|2008|    4|  1|    3|     21| 100|[0.10115931183099...|
|183|         4|2008|    4|  1|    3|     21| 100|[-0.0253764074295...|
+---+----------+----+-----+---+-----+-------+----+--------------------+
only showing top 5 rows


In [3]:
test_vectorized.show(20)

+---+----------+----+-----+---+-----+-------+----+--------------------+
| id|session ID|year|month|day|order|country|next|              vector|
+---+----------+----+-----+---+-----+-------+----+--------------------+
|151|         4|2008|    4|  1|    1|     21| 183|[0.10115931183099...|
|151|         4|2008|    4|  1|    2|     21|  52|[0.10115931183099...|
|183|         4|2008|    4|  1|    2|     21|  52|[-0.0253764074295...|
|151|         4|2008|    4|  1|    3|     21| 100|[0.10115931183099...|
|183|         4|2008|    4|  1|    3|     21| 100|[-0.0253764074295...|
| 52|         4|2008|    4|  1|    3|     21| 100|[-0.0784461349248...|
| 97|         8|2008|    4|  1|    1|      9|  19|[0.41063296794891...|
| 97|         8|2008|    4|  1|    2|      9| 199|[0.41063296794891...|
| 19|         8|2008|    4|  1|    2|      9| 199|[0.45786464214324...|
| 97|         8|2008|    4|  1|    3|      9|  10|[0.41063296794891...|
| 19|         8|2008|    4|  1|    3|      9|  10|[0.45786464214

### Weighted pooling

You can try different decay rate for the experiments below, i have tried 0.01, 0.1, 0.3, 0.5, 0.8, 0.9
Peak performance decay 0.01 (simply KNN lol)

In [4]:
from pyspark.sql import functions as F, Window
from pyspark.ml.stat import Summarizer
decay = 0.01

#Infers original item position in session
w_item = Window.partitionBy("session ID", "id")
test_weighted = (test_vectorized
        .withColumn("item_pos",F.min("order").over(w_item))
        .withColumn("weight",F.pow(F.lit(decay), F.col("order") - F.col("item_pos"))))
sequential_window = Window.partitionBy("session ID", "order")
test_weighted = (test_weighted
        .withColumn("weight_sum", F.sum("weight").over(sequential_window))
        .withColumn("weight", F.col("weight") / F.col("weight_sum")))
pooled_test = test_weighted.groupBy("session ID", "order", "next").agg(Summarizer.sum(F.col("vector"), weightCol=F.col("weight")).alias("pooled"))
pooled_test = pooled_test.join(sequences_test.select("session ID", "order", "sequence"), on = ["session ID", "order"], how = "inner").withColumnRenamed("sequence", "visited")
pooled_test.show()

+----------+-----+----+--------------------+--------------------+
|session ID|order|next|              pooled|             visited|
+----------+-----+----+--------------------+--------------------+
|         4|    1| 183|[0.10115931183099...|               [151]|
|         4|    2|  52|[-0.0241235785260...|          [151, 183]|
|         4|    3| 100|[-0.0779029631402...|      [151, 183, 52]|
|         8|    1|  19|[0.41063296794891...|                [97]|
|         8|    2| 199|[0.45739700180469...|            [97, 19]|
|         8|    3|  10|[0.03872414328487...|       [97, 19, 199]|
|         8|    4|   4|[0.39574674207338...|   [97, 19, 199, 10]|
|         8|    5| 197|[0.45584289406725...|[97, 19, 199, 10, 4]|
|         8|    6| 136|[0.39518508237345...|[97, 19, 199, 10,...|
|         8|    7|  54|[0.27112195401606...|[97, 19, 199, 10,...|
|         8|    8| 212|[0.44632316862634...|[97, 19, 199, 10,...|
|        10|    1|  27|[-0.1562576442956...|               [164]|
|        1

### Raw cosine retrieval

In [ ]:
from pyspark.sql import types as T
from pyspark.sql import Window

def cosine_sim(v1, v2):
    if v1 is None or v2 is None:
        return None
    denom = v1.norm(2) * v2.norm(2)
    if denom == 0:
        return 0.0
    return float(v1.dot(v2) / denom)
N = 10
K = 1 #ignore one past interaction
cosine_udf = F.udf(cosine_sim, T.DoubleType())
w = Window.partitionBy("Session ID").orderBy(F.desc("order"))
full_sequence = pooled_test.withColumn("full", F.row_number().over(w).alias("last")).filter(F.col("last") == 1)

scores = (full_sequence
    .crossJoin(vocab)
    .filter(~F.array_contains(F.slice(F.col("visited"), F.size("visited") + 1 - K, K), F.col("word")))
    .withColumn("score", cosine_udf(F.col("pooled"), F.col("vector"))))

w = Window.partitionBy("session ID").orderBy(F.desc("score"))
scores = scores.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= N)
scores.show()

eval_rows = (
    scores
    .withColumn("hit", (F.col("word") == F.col("next")).cast("int"))
    .withColumn("dcg", F.when(F.col("hit") == 1, 1.0 / F.log2(F.col("rank") + F.lit(1.0))).otherwise(0.0))
)

metrics = (
    eval_rows
    .groupBy("session ID", "order", "next")
    .agg(
        F.max("hit").alias("hit"),
        F.max("dcg").alias("ndcg")
    )
    .agg(
        F.avg("hit").alias(f"recall@{N}"),
        F.avg("ndcg").alias(f"ndcg@{N}")
    )
)
metrics.show()
catalog_size = vocab.select("word").distinct().count()
recommended_items = scores.select("word").distinct().count()

print(f"Catalog coverage: {recommended_items / 217:.4%} ({recommended_items}/{217})")

+----------+-----+----+--------------------+--------------------+----+----+--------------------+------------------+----+
|session ID|order|next|              pooled|             visited|full|word|              vector|             score|rank|
+----------+-----+----+--------------------+--------------------+----+----+--------------------+------------------+----+
|        28|    5| 128|[-0.6166949746103...|[177, 201, 130, 1...|   1|   7|[-0.4964592158794...|0.9578168963971787|   1|
|        28|    5| 128|[-0.6166949746103...|[177, 201, 130, 1...|   1| 156|[-0.5248047709465...|0.9260749820645127|   2|
|        28|    5| 128|[-0.6166949746103...|[177, 201, 130, 1...|   1|  57|[-0.3852210938930...|0.9220828167154218|   3|
|        28|    5| 128|[-0.6166949746103...|[177, 201, 130, 1...|   1|  84|[-0.5560542345046...|0.9187994564695251|   4|
|        28|    5| 128|[-0.6166949746103...|[177, 201, 130, 1...|   1| 187|[-0.4474692642688...|0.9046984178240423|   5|
|        28|    5| 128|[-0.61669

+------------------+-----------------+
|         recall@10|          ndcg@10|
+------------------+-----------------+
|0.3278207522005868|0.162462228164167|
+------------------+-----------------+



### Faster vector search (here not ingoring last click, but uses 30k partial sequences for evaluation)
Scoring might be inflated by double clicks.

In [ ]:
from pyspark.ml.feature import Normalizer, BucketedRandomProjectionLSH
pooled= Normalizer(inputCol="pooled",outputCol="features").transform(pooled_test)
vocab_norm = Normalizer(inputCol="vector", outputCol="features").transform(vocab)
#faster than full cosine evaluation
lsh = BucketedRandomProjectionLSH(inputCol="features",outputCol="hashes",bucketLength=1,numHashTables=5)
lsh_model = lsh.fit(vocab_norm)


candidates = lsh_model.approxSimilarityJoin(pooled, vocab_norm, threshold=1.3, distCol="distance")
w = Window.partitionBy(F.col("datasetA.session ID"), F.col("datasetA.order")).orderBy(F.asc("distance"))
top_n = (
    candidates
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= N)
    .select(
        F.col("datasetA.session ID").alias("session ID"),
        F.col("datasetA.order").alias("order"),
        F.col("datasetA.next").alias("next"),
        F.col("datasetB.word").alias("word"),
        F.col("distance"),
        F.col("rank")
    )
)

top_n.show(20, truncate=False)
eval_rows = (
    top_n
    .withColumn("hit", (F.col("word") == F.col("next")).cast("int"))
    .withColumn("dcg", F.when(F.col("hit") == 1, 1.0 / F.log2(F.col("rank") + F.lit(1.0))).otherwise(0.0))
)

metrics = (
    eval_rows
    .groupBy("session ID", "order", "next")
    .agg(
        F.max("hit").alias("hit"),
        F.max("dcg").alias("ndcg")
    )
    .agg(
        F.avg("hit").alias(f"recall@{N}"),
        F.avg("ndcg").alias(f"ndcg@{N}")
    )
)

metrics.show()

catalog_size = vocab.select("word").distinct().count()
recommended_items = top_n.select("word").distinct().count()

print(f"Catalog coverage: {recommended_items / 217:.4%} ({recommended_items}/{217})")

+----------+-----+----+----+--------------------+----+
|session ID|order|next|word|distance            |rank|
+----------+-----+----+----+--------------------+----+
|8         |3    |10  |199 |0.012234066585752393|1   |
|8         |3    |10  |192 |0.45656369121650703 |2   |
|8         |3    |10  |149 |0.49091432779865546 |3   |
|8         |3    |10  |146 |0.49916695067141476 |4   |
|8         |3    |10  |93  |0.5607130047083964  |5   |
|8         |3    |10  |22  |0.5869271885993587  |6   |
|8         |3    |10  |140 |0.6127666571916667  |7   |
|8         |3    |10  |58  |0.6231356197501171  |8   |
|8         |3    |10  |52  |0.6318267994695637  |9   |
|8         |3    |10  |110 |0.6436102482714134  |10  |
|10        |2    |49  |27  |0.011429931735627386|1   |
|10        |2    |49  |148 |0.3398172923592018  |2   |
|10        |2    |49  |105 |0.3554365536723103  |3   |
|10        |2    |49  |205 |0.4010424081364441  |4   |
|10        |2    |49  |53  |0.4486359228729488  |5   |
|10       

+------------------+-------------------+
|         recall@10|            ndcg@10|
+------------------+-------------------+
|0.4557238514699037|0.22105508034739613|
+------------------+-------------------+



Catalog coverage: 98.1567% (213/217)


### Simple Markov (first order) (NOT USED)

In [ ]:
df = load_data(data_dir)
sequences, tokens = tokenize(df, build_sequences=False)

TOKENIZED DF:
+----+-----+---+-----+-------+----------+---+
|year|month|day|order|country|session ID| id|
+----+-----+---+-----+-------+----------+---+
|2008|    4|  1|    1|     29|         1|197|
|2008|    4|  1|    2|     29|         1|149|
|2008|    4|  1|    3|     29|         1| 51|
|2008|    4|  1|    4|     29|         1| 91|
|2008|    4|  1|    5|     29|         1|111|
|2008|    4|  1|    6|     29|         1|193|
|2008|    4|  1|    7|     29|         1| 73|
|2008|    4|  1|    8|     29|         1| 97|
|2008|    4|  1|    9|     29|         1|118|
|2008|    4|  1|    1|     29|         2| 87|
+----+-----+---+-----+-------+----------+---+
only showing top 10 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+----

### Constructing transitions

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
w = Window.partitionBy("session ID").orderBy("order")
transitions = sequences.select(F.col("id").alias("src"), "session ID", "order", F.lag("id", offset = -1).over(w).alias("dst"))
transitions = transitions.dropna()
transitions = transitions.groupBy("src", "dst").count().orderBy("count", ascending = False)
w = Window.partitionBy("src")
transitions = transitions.withColumn("weight", F.col("count") / F.sum("count").over(w)).drop("count")
transitions = transitions.filter(transitions["src"] != transitions["dst"])
print("Total transitions (excluding self-connections): ", transitions.count())
transitions.show(5)

Total transitions (excluding self-connections):  20295
+---+---+--------------------+
|src|dst|              weight|
+---+---+--------------------+
|  0|165| 0.09536784741144415|
|  0|203| 0.05722070844686648|
|  0|126|0.051771117166212535|
|  0|113|0.051771117166212535|
|  0|114|0.035422343324250684|
+---+---+--------------------+
only showing top 5 rows
